In [5]:
import io
import os
import time
import zipfile
import datetime
import requests
import traceback
import pandas as pd

In [2]:
DATA_FOLDER = "gdelt_csvs"
os.makedirs(DATA_FOLDER, exist_ok=True)

In [3]:
def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + datetime.timedelta(n)

In [19]:
def download_and_filter_gdelt(date):
    date_str = date.strftime("%Y%m%d")
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    try:
        print(f"Downloading {url} ...")
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"No data for {date_str} (status {r.status_code})")
            return False

        z = zipfile.ZipFile(io.BytesIO(r.content))
        filename = z.namelist()[0]
        with z.open(filename) as f:
            columns = [
                "GKGRECORDID","V2DATE","V2SOURCECOLLECTIONIDENTIFIER","V2SOURCECOMMONNAME",
                "DocumentIdentifier","Counts","V2Counts","Themes","V2Themes","Locations","V2Locations",
                "Persons","V2Persons","Organizations","V2Organizations","V2Tone","Dates","GCAM",
                "SharingImage","RelatedImages","SocialImageEmbeds","SocialVideoEmbeds","Quotations",
                "AllNames","Amounts","TranslationInfo","ExtrasXML"
            ]
            df = pd.read_csv(f, sep="\t", header=None, names=columns, dtype=str)

        
        print(df[['V2Themes','DocumentIdentifier']].head(5))

        
        keyword_filter = (
            (
                df['V2Themes'].str.contains("GOLD", case=False, na=False) |
                df['DocumentIdentifier'].str.contains("gold", case=False, na=False) |
                df['DocumentIdentifier'].str.contains("bullion", case=False, na=False)
            )
            &
            (
                df['V2Themes'].str.contains("ECON", case=False, na=False) |
                df['V2Themes'].str.contains("FINANCE", case=False, na=False) |
                df['V2Themes'].str.contains("COMMODITY", case=False, na=False) |
                df['V2Themes'].str.contains("MARKET", case=False, na=False)
            )
        )

        filtered_df = df[keyword_filter]

        output_path = os.path.join(DATA_FOLDER, f"{date_str}_gold_filtered.csv")

        if not filtered_df.empty:
            filtered_df.to_csv(output_path, index=False)
            print(f"✅ Saved {len(filtered_df)} gold-finance news for {date_str}")
        else:
            pd.DataFrame([{"Date": date_str, "Note": "No gold market news"}]).to_csv(output_path, index=False)
            print(f"⚠️ No gold market news found for {date_str}")

        return True

    except Exception as e:
        print(f"❌ Error processing {date_str}: {e}")
        return False


In [ ]:
start_date = datetime.date(2014, 1, 1)
end_date = datetime.date.today()

for single_date in daterange(start_date, end_date):
    download_and_filter_gdelt(single_date)
    time.sleep(1)  

                                            V2Themes  \
0                                      CAMEOEVENTIDS   
1                                                NaN   
2  281472124,281497660,281472095,281497659,281472...   
3                                                NaN   
4                                                NaN   

                                  DocumentIdentifier  
0                                          LOCATIONS  
1  1#Iran#IR#IR#32#53#IR;2#Washington, United Sta...  
2  1#India#IN#IN#20#77#IN;4#Kolkata, West Bengal,...  
3  1#United Kingdom#UK#UK#54#-2#UK;1#Ireland#EI#E...  
4  2#Utah, United States#US#USUT#40.1135#-111.854...  
⚠️ No gold market news found for 20140101
                                  V2Themes  \
0                            CAMEOEVENTIDS   
1                                      NaN   
2                                      NaN   
3  281590303,281535049,281590473,281590474   
4                                      NaN   

              